# Intro 1: querying Hetionet with Cypher

Hetionet is the knowledge graph our whole project runs on. Its authors host it on a public Neo4j server, so we do not have to install or load anything. We send a Cypher query over HTTP and get JSON back.

This notebook does that once by hand and then with `run_cypher` from `src/db.py`, which is what the rest of our code uses. The last two sections are about `EXPLAIN` and about the server being an old Neo4j 3.5, because both turned out to matter for the benchmark.

In [1]:
import os, sys

# the notebook sits in notebooks/intro, the project code expects to run from the repo root
if not os.path.exists("pyproject.toml"):
    os.chdir("../..")
sys.path.append("src")

## One query by hand

The server has a single address for queries. We POST the Cypher text wrapped in a bit of JSON, and the answer is JSON as well.

In [2]:
import httpx

URL = "https://neo4j.het.io/db/data/transaction/commit"

def ask(cypher):
    response = httpx.post(URL, json={"statements": [{"statement": cypher}]}, timeout=60)
    return response.json()

answer = ask("MATCH (n) RETURN count(n) AS nodes")
answer

{'results': [{'columns': ['nodes'],
   'data': [{'row': [47031], 'meta': [None]}]}],
 'errors': []}

Every reply has `results` and `errors`. The number we asked for is a few levels down:

In [3]:
answer["results"][0]["data"][0]["row"][0]

47031

## The same with `run_cypher`

Digging through that structure for every query gets old quickly, so `src/db.py` wraps the call in a function. `run_cypher` returns the rows as a list of dictionaries, tries again when the server is busy, cuts long results to 50 rows and blocks anything that would write to the graph. It is also the only tool our agents get, so everything a model learns about Hetionet comes through this function.

First a look at what is in the graph. Nodes have a label, which says what kind of thing they are, and relationships have a type.

In [4]:
from db import run_cypher

labels = run_cypher("CALL db.labels()")["rows"]
types = run_cypher("CALL db.relationshipTypes()")["rows"]

print(len(labels), "node labels:", ", ".join(sorted(row["label"] for row in labels)))
print(len(types), "relationship types, for example:", ", ".join(sorted(row["relationshipType"] for row in types)[:5]))

11 node labels: Anatomy, BiologicalProcess, CellularComponent, Compound, Disease, Gene, MolecularFunction, Pathway, PharmacologicClass, SideEffect, Symptom
24 relationship types, for example: ASSOCIATES_DaG, BINDS_CbG, CAUSES_CcSE, COVARIES_GcG, DOWNREGULATES_AdG


The end of a relationship name is an abbreviation of what it connects, so `TREATS_CtD` reads *Compound treats Disease*. With that we can already ask something useful, for example which compounds treat multiple sclerosis.

In [5]:
import pandas as pd

result = run_cypher("""
    MATCH (c:Compound)-[:TREATS_CtD]->(d:Disease {name: 'multiple sclerosis'})
    RETURN c.name AS compound
    ORDER BY compound
""")
print(result["total_rows"], "rows, truncated:", result["truncated"])
pd.DataFrame(result["rows"]).head()

11 rows, truncated: False


,compound
0,Azathioprine
1,Betamethasone
2,Cladribine
3,Dexamethasone
4,Fingolimod


The questions of our benchmark are stored in `data/questions.jsonl`, each together with a reference query and its result. The third one needs two hops and a count:

In [6]:
import json

questions = [json.loads(line) for line in open("data/questions.jsonl", encoding="utf-8")]
q03 = questions[2]
print(q03["question"])
print(q03["gold_cypher"])
pd.DataFrame(run_cypher(q03["gold_cypher"])["rows"])

Which 5 compounds bind the most genes associated with multiple sclerosis? Return name and number of shared genes; break ties alphabetically by name.
MATCH (c:Compound)-[:BINDS_CbG]->(g:Gene)<-[:ASSOCIATES_DaG]-(:Disease {name:'multiple sclerosis'}) RETURN c.name AS name, count(DISTINCT g) AS shared_genes ORDER BY shared_genes DESC, name ASC LIMIT 5


,name,shared_genes
0,Calcidiol,3
1,Ergocalciferol,3
2,Glucosamine,3
3,Alfacalcidol,2
4,Ampicillin,2


A query that tries to change the graph never leaves our machine:

In [7]:
run_cypher("CREATE (n:Test) RETURN n")

{'error': 'Blocked: only read-only Cypher (MATCH/RETURN) is allowed.'}

## EXPLAIN: checking a query without running it

With `EXPLAIN` in front, the server only plans a query and returns no rows. This is useful for us because a model can write perfectly valid Cypher that uses a name Hetionet does not have. Below is the same query twice, once correct and once with `:Drug` instead of `:Compound`.

In [8]:
good = "MATCH (c:Compound)-[:TREATS_CtD]->(d:Disease {name: 'multiple sclerosis'}) RETURN c.name"
bad = good.replace(":Compound", ":Drug")

for query in [good, bad]:
    answer = ask("EXPLAIN " + query)
    print(query)
    print("  errors:", answer["errors"])
    print("  warnings:", [n["title"] for n in answer.get("notifications", [])])

MATCH (c:Compound)-[:TREATS_CtD]->(d:Disease {name: 'multiple sclerosis'}) RETURN c.name
  errors: []
  warnings: []


MATCH (c:Drug)-[:TREATS_CtD]->(d:Disease {name: 'multiple sclerosis'}) RETURN c.name
  errors: []
  warnings: ['The provided label is not in the database.']


So the planner notices the unknown label. Now the same wrong query without `EXPLAIN`:

In [9]:
run_cypher(bad)

{'columns': ['c.name'], 'rows': [], 'total_rows': 0, 'truncated': False}

No error and no warning, only an empty result. For a benchmark this is the worst case, because an empty answer looks like "nothing was found" when the real problem is a label that does not exist. That is why in the preprocessing notebook every generated query is checked before it is run, and `EXPLAIN` is one of the checks.

## The server speaks an old Cypher

The public server runs Neo4j 3.5, and Cypher has changed since then. Language models have mostly seen the newer versions, so they tend to write things like an `EXISTS { ... }` subquery, which only arrived in Neo4j 4:

In [10]:
result = run_cypher("""
    MATCH (c:Compound)
    WHERE EXISTS { MATCH (c)-[:TREATS_CtD]->(:Disease {name: 'multiple sclerosis'}) }
    RETURN c.name AS compound LIMIT 5
""")
print(result["error"])

Invalid input '(': expected whitespace, comment, ':', ',' or '}' (line 3, column 30 (offset: 49))
"    WHERE EXISTS { MATCH (c)-[:TREATS_CtD]->(:Disease {name: 'multiple sclerosis'}) }"
                               ^


The error comes back as text in the result and not as an exception, so an agent can read it and try again. In 3.5 the same condition is written as a plain pattern in `WHERE`:

In [11]:
result = run_cypher("""
    MATCH (c:Compound)
    WHERE (c)-[:TREATS_CtD]->(:Disease {name: 'multiple sclerosis'})
    RETURN c.name AS compound ORDER BY compound LIMIT 5
""")
[row["compound"] for row in result["rows"]]

['Azathioprine', 'Betamethasone', 'Cladribine', 'Dexamethasone', 'Fingolimod']

That is everything we need from the database for now. From here on all our code talks to Hetionet through `run_cypher`, and the system prompt tells the models that they are writing for Neo4j 3.5.